# XGBoost with expanded pre-election predictors

Copy of the modelling approach in `Models/xgboost_model.py`, expanded to every column labelled **Predictor** in `TEST_TRAIN/predictor_descriptions.md`. Identifiers, election metadata and current outcomes are excluded from model inputs; `winner` is used only as the target.

The original grid (16 combinations), five-fold stratified accuracy scoring, fold-local preprocessing and party-label encoding are retained. Train on elections up to and including 2017, then predict the held-out 2019 election. Internal pooled cross-validation is not temporal; 2019 is evaluated only after parameter selection. The 2024 test.csv is not used.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.utils.validation import check_is_fitted
from xgboost import XGBClassifier

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "TEST_TRAIN" / "train.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository.")

data = pd.read_csv(ROOT / "TEST_TRAIN" / "train.csv")
election_year = pd.to_numeric(data["election"], errors="raise")
train = data.loc[election_year <= 2017].copy()
test = data.loc[election_year == 2019].copy()
print("Training elections:", sorted(train["election"].unique()))
print("Prediction election: 2019")


Training elections: [np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2001), np.int64(2005), np.int64(2010), np.int64(2015), np.int64(2017)]
Prediction election: 2019


In [2]:
FEATURE_COLUMNS = [
    "country/region",
    "previous_con_national_vote_share",
    "previous_lab_national_vote_share",
    "previous_lib_national_vote_share",
    "previous_natSW_national_vote_share",
    "previous_oth_national_vote_share",
    "previous_winning_party_last_election_vote_share",
    "previous_second_party_last_election_vote_share",
    "previous_winner",
    "previous_con_share",
    "previous_lib_share",
    "previous_lab_share",
    "previous_natSW_share",
    "con_polling",
    "lab_polling",
    "lib_polling",
    "incumbent",
    "supported_incumbent",
    "incumbent_polling",
    "opposition_polling",
    "con_national_change",
    "lab_national_change",
    "lib_national_change",
    "natSW_national_change",
    "oth_national_change",
    "previous_margin_1st_2nd",
    "projected_con_share",
    "projected_lib_share",
    "projected_lab_share",
    "holder_national_swing",
    "challenger_national_swing",
    "holder_polling",
    "challenger_polling",
]
CATEGORICAL_COLUMNS = ["country/region", "previous_winner", "incumbent"]
NUMERIC_COLUMNS = [
    column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS
]

for name, frame in [("train", train), ("test", test)]:
    missing = set(FEATURE_COLUMNS + ["winner"]) - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

training = train.dropna(subset=["winner"]).copy()
testing = test.dropna(subset=["winner"]).copy()
if training.empty or testing.empty:
    raise ValueError("Training and test data must both contain labelled rows.")
if training["winner"].value_counts().min() < 5:
    raise ValueError("Each training party needs at least five rows for five-fold CV.")

display(pd.DataFrame({
    "train_missing_fraction": training[FEATURE_COLUMNS].isna().mean(),
    "test_missing_fraction": testing[FEATURE_COLUMNS].isna().mean(),
}))
print(f"{len(FEATURE_COLUMNS)} predictors; {len(training)} training rows; "
      f"{len(testing)} labelled test rows.")
print("Entirely missing training predictors:",
      training[FEATURE_COLUMNS].columns[
          training[FEATURE_COLUMNS].isna().all()
      ].tolist())


,train_missing_fraction,test_missing_fraction
country/region,0.000000,0.000000
previous_con_national_vote_share,0.000000,0.000000
previous_lab_national_vote_share,0.000000,0.000000
previous_lib_national_vote_share,0.000000,0.000000
previous_natSW_national_vote_share,0.000000,0.000000
previous_oth_national_vote_share,0.000000,0.000000
previous_winning_party_last_election_vote_share,0.018135,0.000000
previous_second_party_last_election_vote_share,0.018529,0.001582
previous_winner,0.018135,0.000000
previous_con_share,0.018924,0.001582


33 predictors; 5073 training rows; 632 labelled test rows.
Entirely missing training predictors: ['natSW_national_change', 'oth_national_change']


## Model and tuning

Numeric missing values pass through to XGBoost. Categorical predictors are one-hot encoded inside each cross-validation fold, with unseen categories ignored. `natSW_national_change` and `oth_national_change` are retained from the guide but have no polling inputs, so they currently contain no usable signal. All target parties, including `oth`, are retained.

In [3]:
class LabelledXGBClassifier(ClassifierMixin, BaseEstimator):
    """Keep target encoding local to each fitted estimator."""

    def __init__(self, n_estimators=100, max_depth=3):
        self.n_estimators = n_estimators
        self.max_depth = max_depth

    def fit(self, X, y):
        self.label_encoder_ = LabelEncoder()
        encoded = self.label_encoder_.fit_transform(y)
        self.classes_ = self.label_encoder_.classes_
        self.model_ = XGBClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            objective="multi:softprob",
            num_class=len(self.classes_),
            eval_metric="mlogloss",
            n_jobs=1,
            random_state=42,
        )
        self.model_.fit(X, encoded)
        self.n_features_in_ = self.model_.n_features_in_
        return self

    def predict(self, X):
        check_is_fitted(self, "model_")
        encoded = self.model_.predict(X).astype(int)
        return self.label_encoder_.inverse_transform(encoded)

    def predict_proba(self, X):
        check_is_fitted(self, "model_")
        return self.model_.predict_proba(X)


preprocessing = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     CATEGORICAL_COLUMNS),
    ("numeric", "passthrough", NUMERIC_COLUMNS),
])
pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", LabelledXGBClassifier()),
])
search = GridSearchCV(
    estimator=pipeline,
    param_grid={
        "classifier__n_estimators": [20, 35, 50, 100],
        "classifier__max_depth": [2, 3, 4, 5],
    },
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    error_score="raise",
)
search.fit(training[FEATURE_COLUMNS], training["winner"])
model = search.best_estimator_
print("Best parameters:", search.best_params_)
print(f"Best mean CV accuracy: {search.best_score_:.3%}")
display(pd.DataFrame(search.cv_results_)[[
    "param_classifier__n_estimators",
    "param_classifier__max_depth",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].sort_values("rank_test_score"))


Best parameters: {'classifier__max_depth': 2, 'classifier__n_estimators': 20}
Best mean CV accuracy: 87.127%


,param_classifier__n_estimators,param_classifier__max_depth,mean_test_score,std_test_score,rank_test_score
0,20,2,0.871274,0.021949,1
1,35,2,0.862802,0.024808,2
8,20,4,0.860043,0.035554,3
4,20,3,0.859253,0.031835,4
12,20,5,0.855906,0.038587,5
2,50,2,0.852749,0.031627,6
9,35,4,0.852161,0.043729,7
5,35,3,0.851764,0.038612,8
13,35,5,0.848220,0.039573,9
3,100,2,0.847822,0.040236,10


## Held-out evaluation

Election and constituency names below are reporting fields only. Changed-seat accuracy measures rows where the actual winner differs from the previous winner, excluding rows with an unknown previous winner. Repeatedly choosing features using these test results would make this test set part of model selection.

In [4]:
predictions = model.predict(testing[FEATURE_COLUMNS])
print(f"Test accuracy: {accuracy_score(testing['winner'], predictions):.3%}")
print(classification_report(testing["winner"], predictions, zero_division=0))

labels = sorted(set(testing["winner"]) | set(predictions))
display(pd.DataFrame(
    confusion_matrix(testing["winner"], predictions, labels=labels),
    index=pd.Index(labels, name="actual"),
    columns=pd.Index(labels, name="predicted"),
))

report_columns = [
    column for column in ["election", "constituency_name", "previous_winner", "winner"]
    if column in testing.columns
]
results = testing[report_columns].copy()
results["prediction"] = predictions
results["correct"] = results["winner"].eq(results["prediction"])
changed = testing["previous_winner"].notna() & testing["winner"].ne(
    testing["previous_winner"]
)
if changed.any():
    print(f"Changed-seat accuracy: {results.loc[changed, 'correct'].mean():.3%} "
          f"({changed.sum()} rows)")
else:
    print("No labelled changed seats available for evaluation.")
if "election" in results:
    display(results.groupby("election").agg(
        rows=("correct", "size"), accuracy=("correct", "mean")
    ))
display(results.head(20))


Test accuracy: 90.506%
              precision    recall  f1-score   support

         con       0.94      0.92      0.93       365
         lab       0.86      0.94      0.90       202
         lib       0.56      0.82      0.67        11
       natSW       1.00      0.75      0.86        52
         oth       0.50      0.50      0.50         2

    accuracy                           0.91       632
   macro avg       0.77      0.78      0.77       632
weighted avg       0.91      0.91      0.91       632



predicted,con,lab,lib,natSW,oth
actual,,,,,
con,334,26,4,0,1
lab,11,189,2,0,0
lib,2,0,9,0,0
natSW,8,4,1,39,0
oth,0,1,0,0,1


Changed-seat accuracy: 38.158% (76 rows)


,rows,accuracy
election,,
2019,632,0.905063


,election,constituency_name,previous_winner,winner,prediction,correct
4844,2019,Aberavon,lab,lab,lab,True
4845,2019,Aberconwy,con,con,con,True
4846,2019,Aberdeen North,natSW,natSW,natSW,True
4847,2019,Aberdeen South,con,natSW,con,False
4848,2019,Airdrie & Shotts,natSW,natSW,natSW,True
4849,2019,Aldershot,con,con,con,True
4850,2019,Aldridge-Brownhills,con,con,con,True
4851,2019,Altrincham & Sale West,con,con,con,True
4852,2019,Alyn & Deeside,lab,lab,lab,True
4853,2019,Amber Valley,con,con,con,True


## Feature importance

These are XGBoost's built-in importances for the transformed features. Related polling, swing and share predictors can divide importance between them; these values do not establish causal effects.

In [5]:
feature_names = model.named_steps["preprocessing"].get_feature_names_out()
importances = pd.Series(
    model.named_steps["classifier"].model_.feature_importances_,
    index=feature_names,
    name="importance",
).sort_values(ascending=False)
display(importances.head(30).to_frame())

assert model.named_steps["preprocessing"].feature_names_in_.tolist() == FEATURE_COLUMNS
assert np.isfinite(model.predict_proba(testing[FEATURE_COLUMNS])).all()


,importance
categorical__previous_winner_lib,0.428113
categorical__previous_winner_natSW,0.347383
numeric__projected_lab_share,0.052070
numeric__projected_con_share,0.047356
categorical__previous_winner_lab,0.027676
numeric__projected_lib_share,0.014545
numeric__lib_national_change,0.014435
numeric__previous_natSW_share,0.007697
numeric__previous_lib_national_vote_share,0.006887
numeric__previous_oth_national_vote_share,0.006808
